In [1]:
%run ../module7-genai-langchain/initialize_environment.py

Environment initializing completed successfully.


## Summarize messages

In [5]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

llm = create_azure_llm("chat")

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("tokens", 100),
            keep=("messages", 2)
        )
    ],
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [6]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\nThe user is seeking fictional information about the moon, specifically about its capital, weather, population of cheese miners, and labor relations.\n\n## SUMMARY\n- The moon's capital is identified as Lunapolis (a fictional city).\n- The weather in Lunapolis is described as clear skies, with extreme temperatures ranging from 120°C high to -100°C low.\n- Lunapolis is fictionalized to have a population of 100,000 cheese miners.\n- The user inquires about the likelihood of a strike by the cheese miners' union; no response was given yet.\n\n## ARTIFACTS\nNone\n\n## NEXT STEPS\n- Provide an assessment or speculation on whether the cheese miners' union in Lunapolis might strike, based on the fictional context.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='83129a8e-68af-4a42-840f-a77ec9b19e9f'),
              AIMessage(content='Yes, because they are unhappy

In [7]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT
The user is seeking fictional information about the moon, specifically about its capital, weather, population of cheese miners, and labor relations.

## SUMMARY
- The moon's capital is identified as Lunapolis (a fictional city).
- The weather in Lunapolis is described as clear skies, with extreme temperatures ranging from 120°C high to -100°C low.
- Lunapolis is fictionalized to have a population of 100,000 cheese miners.
- The user inquires about the likelihood of a strike by the cheese miners' union; no response was given yet.

## ARTIFACTS
None

## NEXT STEPS
- Provide an assessment or speculation on whether the cheese miners' union in Lunapolis might strike, based on the fictional context.


## Trim/delete messages

In [8]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [9]:
llm = create_azure_llm()

agent = create_agent(
    model=llm,
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [10]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='35f7a049-1da1-49cd-95f1-3c304acbfc28'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='be8c70ab-17c3-44a5-91d3-596d74d1c968', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='868506ba-5b08-4e6b-9dba-2385f32cc00f'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='0f2b200f-2d0b-45b0-9fec-4e1234f6250a', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='b7ff3e7d-1734-4a7d-997a-2a5687ef466a'),
              AIMessage(content='Could you please check the temperature of the device? Is it un

In [11]:
print(response["messages"][-1].content)

Could you please check the temperature of the device? Is it unusually hot, warm, or cool to the touch? This information can help determine if overheating might be causing the issue.
